# 00 Penguins Data Analysis

This notebook does not train a model yet. It answers three questions: what the data looks like, where missing values appear, and whether the different penguin species show clear feature differences.

The next three notebooks use the same dataset to predict `species`.


In [ ]:
from pathlib import Path
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
TEST_SIZE = 0.25


## 1. Load Data

The notebook tries both local project paths and Jupyter container paths so it can run in either environment.


In [ ]:
candidate_paths = [
    Path('../../data/ml_data/penguins.csv'),
    Path('../data/ml_data/penguins.csv'),
    Path('/workspace/data/ml_data/penguins.csv'),
    Path('/Users/nancy/Desktop/cancer-predictor/data/ml_data/penguins.csv'),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Cannot find penguins.csv. Please check data/ml_data/penguins.csv')

penguins = pd.read_csv(data_path)
print('Data path:', data_path)
print('Data shape:', penguins.shape)
penguins.head()


## 2. Data Structure and Missing Values

First check data quality: column types, missing-value counts, and target-class distribution. The penguin dataset is small, so class counts and missing values can affect model stability.


In [ ]:
print('Column types:')
print(penguins.dtypes)

print('\nMissing-value counts:')
print(penguins.isna().sum())

print('\nspecies class distribution:')
species_counts = penguins['species'].value_counts().sort_index()
print(species_counts)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=species_counts.index, y=species_counts.values, hue=species_counts.index, palette='Set2', legend=False, ax=ax)
ax.set_title('Species distribution')
ax.set_xlabel('Species')
ax.set_ylabel('Count')
plt.show()


## 3. Numeric Feature Distributions

These measurements are usually the core signals for separating penguin species: bill length, bill depth, flipper length, and body mass.


In [ ]:
numeric_columns = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.ravel(), numeric_columns):
    sns.histplot(data=penguins, x=col, hue='species', kde=True, element='step', ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

penguins.groupby('species')[numeric_columns].agg(['count', 'mean', 'std']).round(2)


## 4. Feature Relationships

Scatter plots help us judge whether the model may learn clear decision boundaries. Here we focus on bill length versus bill depth and flipper length versus body mass.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=penguins, x='bill_length_mm', y='bill_depth_mm', hue='species', style='sex', ax=axes[0])
axes[0].set_title('Bill length vs bill depth')

sns.scatterplot(data=penguins, x='flipper_length_mm', y='body_mass_g', hue='species', style='sex', ax=axes[1])
axes[1].set_title('Flipper length vs body mass')
plt.tight_layout()
plt.show()


## 5. Summary

This dataset has 344 rows and 3 species classes. The numeric measurement columns have a few missing values, and `sex` has more missing values. Later modeling notebooks impute missing values inside the pipeline with medians and modes to avoid manual preprocessing leakage.
